# Example images at four levels of processing

Builds presentation-quality stills of one field, by **block** and **odor**, for
whichever groups you name. The four levels are the same field with
progressively more analysis applied:

| level | what it shows |
| --- | --- |
| `fluorescence` | mean raw fluorescence over the pre-odor frames. No normalization at all. |
| `roi_outline` | the same image with curated ROI boundaries drawn on it. |
| `pixel_z` | odor-period z computed independently at every pixel, reduced across trials. |
| `roi_z` | the final analysis units painted with their own odor-period z. |

Levels 3 and 4 are built from the *same* trials with the *same* window, so the
difference between them is attributable to ROI definition rather than to
normalization or trial choice. That is the point of showing them adjacent.

`block` is the manifest's state level: `pre` is awake, `post` is after ket/xyl.

**Cost.** Levels 1 and 3 reread the motion-corrected TIFFs for every selected
trial, which is the slow step on a network-mounted imaging root. Levels 2 and 4
come from products already on disk. Set `PIXEL = False` while you are choosing
groups, odors, and crops, then turn it back on for the final pass.

**On the ROI overlay.** The spatial QC pages already show ROIs over a reference
image — `*_10x_spatialqc.png` and `*_10x_groups.png` at 10x, and the
groups/somas/processes panels of `*_20x_spatialqc.png` at 20x. Those are QC
pages: multi-panel, titled, and coloured by baseline SNR. This notebook's
`roi_outline` differs in two ways worth knowing. It is a bare slide-ready
panel, and its background is the mean fluorescence *of the trials you
selected*, so an awake panel and a ket/xyl panel of the same field genuinely
differ — which matters given how much ket/xyl lowers tonic F. If you would
rather have exactly the QC background, `PIXEL = False` uses the published mask
bundle's reference image, which is the same image the QC pages draw on, and
costs no movie reads.

## 1. Repository bootstrap and explicit inputs

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
HERE = Path.cwd().resolve()
REPO = next((p for p in (HERE, *HERE.parents) if (p / 'analysis').is_dir()), None)
if REPO is None: raise RuntimeError('Could not locate the ODyn-analysis repository root.')
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
from analysis.figures.paths import imaging_root, repo_path
MANIFEST = repo_path('analysis', 'stage0', 'ketxyl_16odor_session_manifest.csv')
IMAGING_ROOT = imaging_root()  # set ODYN_IMAGING_ROOT for this computer/server
OUTPUT = repo_path('analysis', 'figures', 'example_images_outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)
print('Imaging root:', IMAGING_ROOT)
print('Output:', OUTPUT)

## 2. Choose the groups, blocks, odors, and windows

Edit this cell and rerun from here. `GROUPS` are manifest `group_id` values.
`POPULATION` selects which analysis units the ROI levels use: `units` is the
only 10x population; at 20x choose `groups`, `somas`, or `processes`.

**`WINDOW`** picks which frames the response is measured over — `'odor'`,
`'early'`, `'late'`, `'post_odor'`, any `(start, stop)` pair in seconds from
odor onset, or `None` for each trial's exact recorded valve frames. The names
match the epochs the rest of the pipeline measures, so a panel and a summary
statistic mean the same thing.

**`BASELINE`** is independent of `WINDOW` and defaults to every pre-odor frame.
A post-odor panel is therefore still referenced to the same quiet period as the
odor panel, and the two are directly comparable. A window that runs past the
end of the acquisition is rejected rather than silently truncated.

**`SIGMA_PX`** smooths each frame before the pixel z is computed — before, not
after, so the z stays a real z rather than a blurred ratio. Read the caveat in
section 6 before using it: it changes what the numbers mean.

`LIMITS` fixes the diverging colour scale. Keep it identical across every panel
you intend to put side by side on a slide — a per-panel autoscale makes fields
look more similar than they are.

In [ ]:
GROUPS = [214, 215, 219]        # manifest group_id values
BLOCKS = ['pre']                # 'pre' is awake, 'post' is ket/xyl
ODORS = [1, 17, 18]             # odor_id values; see the odor dictionary below
POPULATION = None               # None -> units at 10x, groups at 20x
WINDOW = 'odor'                 # name, (start, stop) in s, or None for valve frames
BASELINE = None                 # None = every pre-odor frame; does not move with WINDOW
SIGMA_PX = 0.0                  # Gaussian smoothing of the frames before the pixel z
REDUCER = 'median'              # across trials, within block and odor
LIMITS = (-2.0, 4.0)            # z colour scale, shared by every panel
PIXEL = True                    # False skips the movie reads; ROI levels only
CROP = None                     # (y0, y1, x0, x1) in pixels, or None
from analysis.figures.example_images import WINDOWS, resolve_window, window_label
print('named windows, seconds from odor onset:', WINDOWS)
print('this run:', window_label(resolve_window(WINDOW)))
odors = pd.read_csv(repo_path('analysis', 'stage0', 'odor_dictionary.csv'))
display(odors.loc[odors.odor_id.isin(ODORS), ['odor_id', 'odor_name', 'role', 'chemical_group']])

## 3. Inventory the requested groups

Written before anything is calculated, so a group with no grouped product on
this computer stays visible instead of vanishing from the outputs.

In [ ]:
from analysis.figures.session_data import available_sessions
inventory = pd.DataFrame(available_sessions(MANIFEST, IMAGING_ROOT))
inventory['group_id'] = inventory.group_id.astype(int)
requested = inventory[inventory.group_id.isin(GROUPS)].copy()
requested.to_csv(OUTPUT / 'example_image_inventory.csv', index=False)
missing = sorted(set(GROUPS) - set(requested.group_id))
if missing: print('NOT IN MANIFEST:', missing)
unavailable = sorted(requested.loc[~requested.available, 'group_id'])
if unavailable: print('NO GROUPED PRODUCT:', unavailable)
display(requested[['group_id', 'mouse', 'population', 'objective', 'depth_class', 'available']])

## 4. Open each session once

`load_context` reads the grouped product and the extraction round behind it:
masks, unit membership, trial table, time axis, and the reported micron scale.
Notebooks loop over odors within a session, so these reads are kept out of the
loop — the imaging root is usually a network mount.

The printed blocks and odors are what each session actually contains, which is
the fastest way to catch an odor that was never delivered in a given block.
`reference` reports whether the session published the mask bundle whose image
the QC pages use.

In [ ]:
from analysis.figures.example_images import (available_blocks, available_odors,
                                               load_context, reference_image)
contexts, references = {}, {}
for row in requested.loc[requested.available].to_dict('records'):
    context = load_context(row, IMAGING_ROOT, population=POPULATION)
    contexts[int(row['group_id'])] = context
    references[int(row['group_id'])] = reference_image(context)
    has_reference = 'yes' if references[int(row['group_id'])] is not None else 'no'
    print(f"group {row['group_id']:>4}  {row['mouse']:<6} {context.objective}  "
          f"{context.population:<10} {len(context.members):>4} units  "
          f"{context.labels.shape}  {context.um_per_px} um/px  reference: {has_reference}")
    for block in available_blocks(context):
        print(f"    {block:<5} odors: {available_odors(context, block)}")

## 5. Build the examples

One `ExampleImages` per group, block, and odor. Anything not present in a
session — or any window that will not fit inside the acquisition — is reported
and skipped rather than aborting the whole sweep.

In [ ]:
from tqdm.auto import tqdm
from analysis.figures.example_images import build_example
examples, skipped = {}, []
requests = [(g, b, o) for g in contexts for b in BLOCKS for o in ODORS]
for group_id, block, odor_id in tqdm(requests, desc='examples', unit='panel'):
    try:
        examples[(group_id, block, odor_id)] = build_example(
            contexts[group_id], block=block, odor_id=odor_id, window_s=WINDOW,
            baseline_s=BASELINE, sigma_px=SIGMA_PX, reducer=REDUCER,
            pixel=PIXEL, reference=references[group_id] if not PIXEL else None,
            progress=True)
    except (ValueError, FileNotFoundError) as error:
        skipped.append({'group_id': group_id, 'block': block, 'odor_id': odor_id,
                        'reason': f'{type(error).__name__}: {error}'})
if skipped:
    display(pd.DataFrame(skipped))
print(f'Built {len(examples)} examples, skipped {len(skipped)}.')
if examples:
    one = next(iter(examples.values()))
    print('Provenance:', one.caption(), '| background:', one.fluorescence_source)

## 6. Preview one example as a ladder

Check the contrast, the crop, and the colour limits here before exporting
everything. `gamma` below 1 lifts dim structure that a linear stretch buries
in a field with a few very bright ROIs; it is a display choice and never
touches anything that is measured.

In [ ]:
import matplotlib.pyplot as plt
from analysis.figures.example_images import LEVELS, LEVEL_TITLES, render_level
KEY = next(iter(examples))          # or set it explicitly: (214, 'pre', 17)
example = examples[KEY]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), constrained_layout=True)
for ax, level in zip(axes, LEVELS):
    ax.imshow(render_level(example, level, crop=CROP, limits=LIMITS,
                           background=True, gamma=0.8, outline_width=1),
              interpolation='nearest')
    ax.set(xticks=[], yticks=[], title=LEVEL_TITLES[level])
fig.suptitle(f'{example.context.describe()} - {example.block_label} - '
             f'odor {example.odor_id} - {example.caption()}')
plt.show()

### Rendering options worth knowing

- `background=False` renders the z levels as a flat diverging map with no
  anatomy underneath, matching the existing `spatial_maps_10x` pages.
- `threshold=` sets the z at which a pixel becomes fully opaque over the
  anatomy. It defaults to half the larger colour limit.
- `unit_outlines=False` outlines every raw ROI instead of the joined analysis
  units. At 10x the joins are the analysis unit, so the default is usually right.
- `palette=True` colours each outline by ROI identity rather than one colour.
- `outline_width=2` reads better when the image is projected.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6), constrained_layout=True)
variants = [
    ('over anatomy', dict(background=True)),
    ('flat map', dict(background=False)),
    ('opaque above z=1', dict(background=True, threshold=1.0)),
]
for ax, (title, options) in zip(axes, variants):
    ax.imshow(render_level(example, 'pixel_z', crop=CROP, limits=LIMITS, **options),
              interpolation='nearest')
    ax.set(xticks=[], yticks=[], title=title)
plt.show()

### The smoothing caveat

`SIGMA_PX` does **not** lower the background noise of the z map. The pixel
noise it removes from the numerator is removed from the baseline SD in the
denominator too, and the two cancel — the background z SD is flat in sigma.
What it does is raise the z of spatially coherent signal, which survives the
blur while the per-pixel SD dividing it shrinks. Contrast against the
background improves, sometimes a lot, but the numbers move with it.

Two consequences for a talk:

1. **z values are not comparable across different sigma.** Hold `SIGMA_PX`
   fixed across any panels that share a colour scale. It is recorded in the
   caption, the filename, and the export manifest so a stray panel is visible.
2. **Keep sigma well under the radius of the structure you are showing.** A
   sigma near the ROI radius pulls background in over the boundary and flattens
   the edge, which reads as a larger, weaker glomerulus.

The sweep below makes the tradeoff visible on the current example: watch the
ROI median grow while the background SD stays put.

In [ ]:
from analysis.figures.example_images import pixel_level, select_trials, signed_rgb
from scipy.ndimage import binary_dilation
SIGMA_SWEEP = [0.0, 1.0, 2.0, 4.0]
if PIXEL:
    ctx = example.context
    idx = select_trials(ctx, block=example.block, odor_id=example.odor_id)
    inside = example.unit_labels > 0
    # Background measured clear of the blur: pixels within ~3 sigma of an ROI
    # pick up bleed from it, which is the edge effect, not the noise floor.
    far = ~binary_dilation(inside, iterations=int(np.ceil(3 * max(SIGMA_SWEEP))) or 1)
    fig, axes = plt.subplots(1, len(SIGMA_SWEEP), figsize=(3.2 * len(SIGMA_SWEEP), 3.8),
                             constrained_layout=True)
    for ax, sigma in zip(axes, SIGMA_SWEEP):
        z = pixel_level(ctx, idx, window_s=WINDOW, baseline_s=BASELINE,
                        sigma_px=sigma, reducer=REDUCER, progress=False)[2]
        ax.imshow(signed_rgb(z, limits=LIMITS, background=example.fluorescence,
                             gamma=0.8), interpolation='nearest')
        ax.set(xticks=[], yticks=[],
               title=f'sigma {sigma:g} px\nROI |z| {np.nanmedian(np.abs(z[inside])):.2f}  '
                     f'bg SD {np.nanstd(z[far]):.2f}')
    fig.suptitle('ROI |z| grows with sigma; background SD clear of the ROIs does not')
    plt.show()
else:
    print('Set PIXEL = True to sweep sigma.')

### Comparing windows

The same field measured over different windows against the one pre-odor
baseline. This is the panel for "the response is early and then it is gone".

In [ ]:
WINDOW_SWEEP = ['early', 'late', 'post_odor']
if PIXEL:
    fig, axes = plt.subplots(1, len(WINDOW_SWEEP), figsize=(3.2 * len(WINDOW_SWEEP), 3.8),
                             constrained_layout=True)
    for ax, name in zip(axes, WINDOW_SWEEP):
        z = pixel_level(ctx, idx, window_s=name, baseline_s=BASELINE,
                        sigma_px=SIGMA_PX, reducer=REDUCER, progress=False)[2]
        ax.imshow(signed_rgb(z, limits=LIMITS, background=example.fluorescence,
                             gamma=0.8), interpolation='nearest')
        ax.set(xticks=[], yticks=[], title=window_label(resolve_window(name)))
    fig.suptitle(f'{ctx.describe()} - odor {example.odor_id} - '
                 f'one pre-odor baseline throughout')
    plt.show()
else:
    print('Set PIXEL = True to sweep windows.')

## 7. Export every example

Each example writes four bare PNGs — no axes, no titles, one file pixel per
image pixel — so they can be dropped straight onto a slide, plus the labelled
ladder figure with a scale bar and colourbar, plus an `.npz` of the underlying
arrays. Keeping the arrays means the colour limits, crop, and gamma can all be
changed later without rereading a single movie.

Filenames carry the block, odor, window, and sigma, so panels made under
different settings cannot quietly overwrite each other.

In [ ]:
from analysis.figures.example_images import export_example
written = [export_example(OUTPUT, item, crop=CROP, limits=LIMITS,
                          background=True, gamma=0.8, dpi=300)
           for item in tqdm(examples.values(), desc='export', unit='example')]
manifest = pd.DataFrame(written)
manifest.to_csv(OUTPUT / 'example_image_manifest.csv', index=False)
display(manifest[['group_id', 'mouse', 'block', 'odor_id', 'n_trials', 'window',
                  'sigma_px', 'fluorescence_source', 'um_per_px']])
print('Wrote', len(manifest) * 4, 'bare panels and', len(manifest), 'ladder figures to', OUTPUT)

## 8. Optional: one level across many odors

For a slide that compares odors rather than processing levels. Reuses the
examples already built, so it costs nothing extra.

In [ ]:
LEVEL = 'pixel_z'
GROUP, BLOCK = KEY[0], KEY[1]
panels = [(o, examples[(GROUP, BLOCK, o)]) for o in ODORS if (GROUP, BLOCK, o) in examples]
if panels:
    names = odors.set_index('odor_id').odor_name.to_dict()
    fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.6),
                             constrained_layout=True, squeeze=False)
    for ax, (odor_id, item) in zip(axes.ravel(), panels):
        ax.imshow(render_level(item, LEVEL, crop=CROP, limits=LIMITS, gamma=0.8),
                  interpolation='nearest')
        ax.set(xticks=[], yticks=[],
               title=f'{names.get(odor_id, odor_id)}\nn={item.n_trials}')
    fig.suptitle(f'{contexts[GROUP].describe()} - {LEVEL_TITLES[LEVEL]} - '
                 f'{panels[0][1].block_label} - {panels[0][1].window_label}')
    fig.savefig(OUTPUT / f'group{GROUP}_{BLOCK}_{LEVEL}_by_odor.png', dpi=300,
                bbox_inches='tight')
    plt.show()

## 9. Re-render from saved arrays

Nothing above needs to run again to change how a finished example looks. This
reads one `.npz` back and rewrites a panel at new colour limits.

In [ ]:
from analysis.figures.example_images import save_png
saved = np.load(OUTPUT / f'{examples[KEY].stem()}_arrays.npz')
rgb = signed_rgb(saved['pixel_z'], limits=(-1.5, 3.0),
                 background=saved['fluorescence'], gamma=0.8)
save_png(OUTPUT / 'rerendered_pixel_z.png', rgb)
plt.figure(figsize=(4, 4)); plt.imshow(rgb, interpolation='nearest')
plt.xticks([]); plt.yticks([]); plt.show()

## Equivalent command line

The same sweep without opening a notebook:

```bash
python -m analysis.figures.example_images --groups 214 215 219 --blocks pre --odors 1 17 18
```

Useful flags: `--window post_odor` or `--window 0.5,2` or `--window valve`;
`--baseline -5,-1`; `--sigma-px 1.5`; `--no-pixel` for a fast ROI-only pass;
`--population somas` at 20x; `--limits -1.5 3` to change the colour scale.